# Spotify popularity — pipeline

Notebook de modelado. Todo el feature engineering está en `src/spotify/features.py`;
el análisis exploratorio de cada variable está en `spotify_gshan_irigoin.ipynb` (respaldo).

Estructura:
1. Carga de datos y features de grupo
2. División de `train` en **train_train** (80 %) y **train_test** (20 %)
3. Pipelines de los 6 modelos
4. Ajuste de hiperparámetros con validación cruzada sobre train_train:
   - 4a. `RandomizedSearchCV` para LinearRegression, DecisionTree, LightGBM y CatBoost
   - 4b. **Optuna** para RandomForest y XGBoost
   - 4c. **Ensamble**: promedio de los 3 mejores modelos de árboles
5. Evaluación de cada modelo en train_test
6. Modelo final reentrenado con todo `train` y predicción sobre `test`
   - 6b. Una submission por cada modelo
7. **TabPFN** (sección aparte, opcional): evaluación en train_test, mezcla con el ensamble y submissions

Las secciones 1–6 no usan TabPFN: al terminar la 6 ya están todas las submissions en `outputs/`.

Métrica de decisión: **RMSE** (la métrica de la competición de Kaggle).

In [1]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV, cross_val_score, cross_val_predict
import optuna
from scipy.stats import loguniform, randint, uniform
from scipy.optimize import minimize
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

# Hace visible src/ aunque el .pth del paquete no se cargue
# (en iCloud Drive, macOS marca el .pth como oculto y Python 3.12 lo ignora)
import sys
from pathlib import Path
SRC = Path.cwd() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from spotify.features import (
    add_group_features,
    ROW_FEATURES,
    make_linear_preprocessor,
    make_tree_preprocessor,
)

RANDOM_STATE = 1928

## 1. Carga de datos y features de grupo

`album_size` y `release_size` cuentan tracks usando train y test juntos, así que se calculan
una sola vez, fuera del pipeline. No usan `popularity`, por lo que no hay fuga del target.

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(train.shape, test.shape)   # (65075, 20) (16268, 19)

train, test = add_group_features(train, test)

X = train.drop(columns="popularity")
y = train["popularity"]

(65075, 20) (16268, 19)


## 2. División train_train / train_test

train_test se reserva: no se usa ni para entrenar ni para elegir hiperparámetros,
solo para medir al final cómo generaliza cada modelo.

In [3]:
X_tt, X_th, y_tt, y_th = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("train_train:", X_tt.shape, "| train_test:", X_th.shape)

train_train: (52060, 47) | train_test: (13015, 47)


## 3. Pipelines y espacios de búsqueda

- **LinearRegression** (baseline): features fila a fila → preprocesador lineal → modelo.
  El preprocesador lineal estandariza las variables continuas, añade x² donde la relación con la
  popularidad tiene forma de U invertida, usa deciles y one-hot, y no tiene columnas redundantes
  (matriz de rango completo).
- **Familia de árboles** (DecisionTree, RandomForest, LightGBM, XGBoost, CatBoost): features fila a fila →
  preprocesador de árboles → modelo. El preprocesador de árboles usa las 14 columnas numéricas
  originales tal cual, los géneros en multi-hot, las features de texto del notebook de EDA,
  target encoding de `artists`, `track_name`, `album_name`, (álbum + conjunto de artistas) y de la
  combinación de géneros, target encoding por **artista individual**, estadísticas de popularidad del
  artista (desviación, máximo, mínimo, mediana), conteos de artista, y palabras clave, tamaño y perfil de
  audio medio del álbum. Los árboles no necesitan escalado.

Todo lo que aprende de los datos (medias, deciles, target encodings) está dentro del pipeline,
así que se reajusta en cada partición de la validación cruzada.

En `RandomizedSearchCV`, cada modelo lleva su espacio de búsqueda (distribuciones) y su `n_iter`.
`loguniform` reparte las pruebas por órdenes de magnitud; `randint(a, b)` da enteros entre `a` y `b - 1`;
`uniform(loc, scale)` da reales entre `loc` y `loc + scale`.

### Caché del preprocesamiento (`Pipeline(memory=...)`)

**Problema.** Construir las features (target encoding con cross-fitting, modelo de texto TF-IDF + Ridge, etc.)
tarda unos 17 s por partición y casi todo en un solo núcleo. Sin caché se repite en **cada** ajuste: con
5 particiones × (30 combinaciones de DecisionTree + 8 de LightGBM + 10 de CatBoost + hasta 80 pruebas de
Optuna + ensamble) son unas 600 veces, es decir, 2–3 horas de trabajo idéntico.

**Por qué es trabajo repetido.** Las features de una partición dependen solo de **qué filas** tiene y de la
configuración del preprocesador, **no** de los hiperparámetros del modelo (`num_leaves`, `max_depth`...).
Como las 5 particiones son siempre las mismas (`cv` con `random_state` fijo), solo hay unos pocos cálculos
distintos: las 5 particiones y train_train completo, para cada uno de los dos preprocesadores.

**Cómo funciona.** Con `Pipeline(..., memory=memory)`, antes de ajustar los pasos `rows` y `prep` sklearn calcula
una huella (hash) de los datos de entrada y de los parámetros del paso:
- si ya existe en la caché, carga de disco el resultado y el transformador ajustado, sin recalcular;
- si no, lo calcula y lo guarda.

El último paso (`model`) nunca se guarda en caché: se entrena siempre.

**No cambia los resultados.** La caché solo evita repetir un cálculo determinista (mismo `random_state`); se
comprobó que el RMSE de CV y las predicciones son idénticos con y sin caché. El cross-fitting y la ausencia de
fuga del target siguen igual, porque cada partición tiene su propia huella.

**Cambios en `features.py`.** La huella no detecta cambios en el *código* de los transformadores. Por eso la
carpeta de la caché lleva en el nombre una huella del contenido de `features.py` y de la versión de
scikit-learn: si cualquiera de los dos cambia, se usa una carpeta nueva (vacía) y todo se recalcula.

**Dónde se guarda.** En `~/.cache/spotify-pipeline/` (fuera del proyecto: ocupa ~700 MB por preprocesador y no
debe sincronizarse con iCloud ni subirse a git). Se puede borrar en cualquier momento; solo hará que la
siguiente ejecución vuelva a calcular las features. Las carpetas de versiones antiguas de `features.py` no se
borran solas.

In [4]:
import hashlib
import sklearn
import spotify.features
from joblib import Memory

# Caché de los pasos "rows" y "prep" (ver la explicación de arriba).
# La huella de features.py + versión de sklearn en el nombre evita reutilizar features de código antiguo.
FEATURES_FINGERPRINT = hashlib.sha256(
    Path(spotify.features.__file__).read_bytes() + sklearn.__version__.encode()
).hexdigest()[:12]
CACHE_DIR = Path.home() / ".cache" / "spotify-pipeline" / f"features_{FEATURES_FINGERPRINT}"
memory = Memory(CACHE_DIR, verbose=0)
print("Caché del preprocesamiento:", CACHE_DIR)

def linear_pipe(model):
    return Pipeline([
        ("rows", ROW_FEATURES),
        ("prep", make_linear_preprocessor(RANDOM_STATE)),
        ("model", model),
    ], memory=memory)

def tree_pipe(model):
    return Pipeline([
        ("rows", ROW_FEATURES),
        ("prep", make_tree_preprocessor(RANDOM_STATE)),
        ("model", model),
    ], memory=memory)

# Modelos ajustados con RandomizedSearchCV -> nombre: (pipeline, espacio de búsqueda, n_iter)
random_search_models = {
    "LinearRegression": (linear_pipe(LinearRegression()), {}, 1),   # sin hiperparámetros

    "DecisionTree": (tree_pipe(DecisionTreeRegressor(random_state=RANDOM_STATE)), {
        "model__max_depth": [None] + list(range(5, 41)),
        "model__min_samples_leaf": randint(1, 301),      # entre 1 y 300
        "model__max_features": [None, 0.5, 0.33],
    }, 30),

    "LightGBM": (tree_pipe(LGBMRegressor(subsample_freq=1, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)), {
        "model__n_estimators": randint(200, 1501),       # entre 200 y 1500
        "model__learning_rate": loguniform(0.01, 0.2),
        "model__num_leaves": randint(15, 256),           # entre 15 y 255
        "model__min_child_samples": randint(5, 101),     # entre 5 y 100
        "model__subsample": uniform(0.6, 0.4),           # entre 0.6 y 1.0 (filas por árbol)
        "model__colsample_bytree": uniform(0.5, 0.5),    # entre 0.5 y 1.0 (columnas por árbol)
        "model__reg_lambda": loguniform(1e-3, 10),
    }, 8),

    "CatBoost": (tree_pipe(CatBoostRegressor(random_seed=RANDOM_STATE, thread_count=-1,
                                             verbose=0, allow_writing_files=False)), {
        "model__iterations": randint(300, 1501),         # entre 300 y 1500
        "model__learning_rate": loguniform(0.02, 0.2),
        "model__depth": randint(4, 11),                  # entre 4 y 10
        "model__l2_leaf_reg": loguniform(1, 10),
    }, 10),
}

Caché del preprocesamiento: ~/.cache/spotify-pipeline/features_54243f8e5366


## 4a. Búsqueda aleatoria (CV de 5 particiones sobre train_train)

La métrica de selección es el RMSE. Para LinearRegression no hay hiperparámetros:
la búsqueda solo calcula su error de validación cruzada (`n_iter=1`).

Los modelos que ya usan todos los núcleos por dentro (LightGBM y CatBoost) se buscan con
`n_jobs=1` para no lanzar varios entrenamientos en paralelo que compitan por los mismos núcleos.

In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
MULTITHREADED = {"LightGBM", "CatBoost"}
tuned = {}   # nombre -> {"pipe": pipeline entrenado en train_train, "cv_rmse": ..., "params": ...}

for name, (pipe, space, n_iter) in random_search_models.items():
    t0 = time.time()
    search = RandomizedSearchCV(
        pipe, space, n_iter=n_iter, cv=cv,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_STATE,
        n_jobs=1 if name in MULTITHREADED else -1,
    )
    search.fit(X_tt, y_tt)
    tuned[name] = {"pipe": search.best_estimator_, "cv_rmse": -search.best_score_,
                   "params": search.best_params_}
    print(f"{name:17s} CV RMSE = {-search.best_score_:.3f}   ({time.time() - t0:.0f} s)")
    print(f"{'':17s} {search.best_params_}")

LinearRegression  CV RMSE = 11.512   (5 s)
                  {}


DecisionTree      CV RMSE = 9.417   (86 s)
                  {'model__max_depth': 11, 'model__max_features': 0.5, 'model__min_samples_leaf': 56}


LightGBM          CV RMSE = 8.195   (1021 s)
                  {'model__colsample_bytree': np.float64(0.8963125304316416), 'model__learning_rate': np.float64(0.022247782919033572), 'model__min_child_samples': 43, 'model__n_estimators': 1021, 'model__num_leaves': 198, 'model__reg_lambda': np.float64(0.002327082098144046), 'model__subsample': np.float64(0.8226919925898581)}


CatBoost          CV RMSE = 8.199   (483 s)
                  {'model__depth': 8, 'model__iterations': 1387, 'model__l2_leaf_reg': np.float64(1.9750038035114694), 'model__learning_rate': np.float64(0.041363360445693664)}


## 4b. Optuna para RandomForest y XGBoost

Optuna propone cada combinación de hiperparámetros a partir de las anteriores (muestreador TPE):
concentra las pruebas en las zonas que van dando menor error, en lugar de muestrear al azar.
Cada prueba (*trial*) se evalúa con la **misma CV de 5 particiones sobre train_train**.

- `N_TRIALS`: número máximo de pruebas por modelo. `TIMEOUT_MIN`: límite de tiempo por modelo
  (la búsqueda se detiene en lo que ocurra primero).
- RandomForest usa 200 árboles durante la búsqueda; el modelo final usa 1000 (sección 6).
- En `max_samples`, `None` significa usar el 100 % de las filas en cada árbol.

In [6]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def cv_rmse(pipe):
    """RMSE medio de la CV de 5 particiones sobre train_train."""
    scores = cross_val_score(pipe, X_tt, y_tt, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=1)
    return -scores.mean()

def rf_params(trial):
    return {
        "n_estimators": 200,
        "max_features": trial.suggest_float("max_features", 0.1, 1.0),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_depth": trial.suggest_categorical("max_depth", [None, 15, 20, 30, 40]),
        "max_samples": trial.suggest_categorical("max_samples", [None, 0.5, 0.6, 0.7, 0.8, 0.9]),
    }

def xgb_params(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 300, 3000, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 50, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }

def make_rf(params):
    return tree_pipe(RandomForestRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1))

def make_xgb(params):
    return tree_pipe(XGBRegressor(**params, tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1))

# nombre: (constructor del pipeline, espacio de búsqueda, N_TRIALS, TIMEOUT_MIN)
optuna_models = {
    "RandomForest": (make_rf, rf_params, 30, 30),
    "XGBoost": (make_xgb, xgb_params, 50, 45),
}

studies = {}
for name, (make_pipe, suggest, n_trials, timeout_min) in optuna_models.items():
    def objective(trial):
        params = suggest(trial)
        trial.set_user_attr("params", params)
        return cv_rmse(make_pipe(params))

    t0 = time.time()
    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, timeout=timeout_min * 60)
    studies[name] = study

    best_params = study.best_trial.user_attrs["params"]
    tuned[name] = {"pipe": make_pipe(best_params).fit(X_tt, y_tt),   # mejor combinación, entrenada en train_train
                   "cv_rmse": study.best_value, "params": best_params}
    print(f"{name:17s} CV RMSE = {study.best_value:.3f}   ({len(study.trials)} trials, {time.time() - t0:.0f} s)")
    print(f"{'':17s} {best_params}")

RandomForest      CV RMSE = 8.307   (30 trials, 1859 s)
                  {'n_estimators': 200, 'max_features': 0.5167448171879894, 'min_samples_leaf': 3, 'max_depth': 40, 'max_samples': None}


XGBoost           CV RMSE = 8.122   (38 trials, 2811 s)
                  {'n_estimators': 2987, 'learning_rate': 0.005608739816148828, 'max_depth': 9, 'min_child_weight': 1.1744733196938295, 'subsample': 0.8678525723156119, 'colsample_bytree': 0.6312890006768803, 'reg_lambda': 0.002529281325567037, 'reg_alpha': 3.397857499986614, 'gamma': 0.19955015395248288}


In [7]:
# Las 5 mejores pruebas de cada estudio
for name, study in studies.items():
    print(name)
    display(study.trials_dataframe(attrs=("number", "value", "params", "duration"))
                 .sort_values("value").head(5).round(4))

RandomForest


,number,value,params_max_depth,params_max_features,params_max_samples,params_min_samples_leaf,duration
20,20,8.3074,40.0,0.5167,NaN,3,0 days 00:01:11.514913
18,18,8.3239,40.0,0.5154,NaN,4,0 days 00:01:09.189711
19,19,8.3256,40.0,0.4038,NaN,4,0 days 00:00:55.082881
21,21,8.3260,40.0,0.4607,NaN,4,0 days 00:01:01.583557
29,29,8.3362,40.0,0.5297,NaN,1,0 days 00:01:22.068512


XGBoost


,number,value,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_alpha,params_reg_lambda,params_subsample,duration
35,35,8.1221,0.6313,0.1996,0.0056,9,1.1745,2987,3.3979,0.0025,0.8679,0 days 00:01:48.340660
36,36,8.1240,0.5606,0.2499,0.0112,8,1.5248,2529,3.7146,0.0209,0.8749,0 days 00:01:13.890034
8,8,8.1286,0.5860,1.0238,0.0063,9,1.2890,2854,1.1409,0.0081,0.9222,0 days 00:01:40.502118
11,11,8.1290,0.6943,1.1044,0.0115,9,1.3408,2635,4.3200,0.0029,0.8017,0 days 00:01:39.776199
23,23,8.1292,0.7525,1.1831,0.0093,9,1.3508,2028,0.1757,0.0011,0.8394,0 days 00:01:18.661043


## 4c. Ensamble ponderado de los 3 mejores modelos de árboles

`VotingRegressor` combina las predicciones de los 3 modelos de árboles con menor RMSE de CV, cada uno con sus
mejores hiperparámetros, mediante un **promedio ponderado**.

**Cómo se eligen los pesos (sin tocar train_test):**
1. Con `cross_val_predict` y la **misma CV de 5 particiones** se obtiene la predicción fuera de muestra (OOF) de
   cada modelo sobre train_train.
2. Se buscan los pesos (no negativos y que suman 1) que minimizan el RMSE de CV de la combinación.
3. Si no mejoran a los pesos iguales, se usan pesos iguales.

Esos mismos OOF dan el RMSE de CV del ensamble: es idéntico a `cross_val_score` sobre el `VotingRegressor`
ponderado y no requiere entrenar más modelos. Con solo 3 pesos estimados sobre ~52.000 filas, el sesgo optimista
de ese CV es despreciable; la comparación limpia es la de train_test, que no interviene en la elección.

In [8]:
TREE_MODELS = ["DecisionTree", "RandomForest", "LightGBM", "XGBoost", "CatBoost"]
top3 = sorted(TREE_MODELS, key=lambda n: tuned[n]["cv_rmse"])[:3]
print("Miembros del ensamble:", top3)

# Predicciones fuera de muestra (OOF) de cada miembro, con las mismas 5 particiones de la CV
t0 = time.time()
P = np.column_stack([cross_val_predict(clone(tuned[n]["pipe"]), X_tt, y_tt, cv=cv, n_jobs=1) for n in top3])
folds = [test_idx for _, test_idx in cv.split(X_tt)]

def oof_cv_rmse(w):
    """CV RMSE (media de las 5 particiones) del promedio ponderado de los miembros."""
    p = P @ w
    return np.mean([root_mean_squared_error(y_tt.iloc[i], p[i]) for i in folds])

# Pesos que minimizan el RMSE fuera de muestra: no negativos y que suman 1
res = minimize(oof_cv_rmse, np.full(3, 1 / 3), method="SLSQP", bounds=[(0, 1)] * 3,
               constraints={"type": "eq", "fun": lambda w: w.sum() - 1})
w_equal, w_opt = np.full(3, 1 / 3), res.x
weights = w_opt if oof_cv_rmse(w_opt) < oof_cv_rmse(w_equal) else w_equal
print(f"Pesos iguales     CV RMSE = {oof_cv_rmse(w_equal):.3f}")
print(f"Pesos óptimos     CV RMSE = {oof_cv_rmse(w_opt):.3f}   " +
      ", ".join(f"{n} {w:.2f}" for n, w in zip(top3, w_opt)))

ensemble = VotingRegressor([(n, clone(tuned[n]["pipe"])) for n in top3], weights=list(weights))
ens_cv = oof_cv_rmse(weights)
tuned["Ensemble"] = {"pipe": clone(ensemble).fit(X_tt, y_tt), "cv_rmse": ens_cv,
                     "params": {"members": top3, "weights": dict(zip(top3, np.round(weights, 3)))}}
print(f"Ensemble          CV RMSE = {ens_cv:.3f}   ({time.time() - t0:.0f} s)")

Miembros del ensamble: ['XGBoost', 'LightGBM', 'CatBoost']


Pesos iguales     CV RMSE = 8.102
Pesos óptimos     CV RMSE = 8.099   XGBoost 0.52, LightGBM 0.25, CatBoost 0.24


Ensemble          CV RMSE = 8.099   (493 s)


## 5. Evaluación en train_test

Se decide con el **RMSE**. El MAE (error medio en puntos de popularidad) y el R² (fracción de varianza
explicada) se muestran solo como referencia para interpretar el resultado.

In [9]:
rows = []
for name, t in tuned.items():
    pred = np.clip(t["pipe"].predict(X_th), 0, 100)   # la popularidad va de 0 a 100
    rows.append({
        "model": name,
        "CV_RMSE (train_train)": t["cv_rmse"],
        "RMSE (train_test)": root_mean_squared_error(y_th, pred),
        "MAE (train_test)": mean_absolute_error(y_th, pred),
        "R2 (train_test)": r2_score(y_th, pred),
    })

results = pd.DataFrame(rows).set_index("model").sort_values("CV_RMSE (train_train)")
results.round(4)

,CV_RMSE (train_train),RMSE (train_test),MAE (train_test),R2 (train_test)
model,,,,
Ensemble,8.0989,7.9188,5.0596,0.8399
XGBoost,8.1221,7.9430,5.0801,0.8389
LightGBM,8.1955,8.0151,5.0964,0.8360
CatBoost,8.1988,7.9943,5.1640,0.8368
RandomForest,8.3074,8.1362,5.1667,0.8310
DecisionTree,9.4170,9.0840,5.7819,0.7893
LinearRegression,11.5118,11.2929,8.4227,0.6744


## 6. Modelo final y predicción sobre test

1. Se toma el modelo con menor RMSE de validación cruzada (`best_cv`).
2. Se reentrena **desde cero** (`clone`) con **todo** `train` y sus mejores hiperparámetros.

Si el modelo final es RandomForest (o el ensamble lo incluye), el ajuste final usa 1000 árboles (la búsqueda
usó 200: más árboles apenas cambian el resultado y hacen la búsqueda 5 veces más lenta).

**Archivos de salida** (carpeta `outputs/`): `submission_<modelo>_<n>.csv`, donde `n` se numera solo (1, 2, 3...)
para no sobrescribir pruebas anteriores. Cada archivo queda anotado en `outputs/submissions_log.csv` con su
RMSE de CV y de train_test, la fecha y los parámetros. La celda imprime **cuál subir a Kaggle**
(la sección 7 puede cambiar esa recomendación si la mezcla con TabPFN es mejor).

In [10]:
import re
from datetime import datetime

OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)
TAGS = {"LinearRegression": "lr", "DecisionTree": "dt", "RandomForest": "rf", "LightGBM": "lgbm",
        "XGBoost": "xgb", "CatBoost": "cat", "Ensemble": "ens", "TabPFN": "tabpfn",
        "Ensemble + TabPFN": "ens_tabpfn"}
sample = pd.read_csv("data/sample_submission.csv")

def save_submission(pred, name, table):
    """Guarda outputs/submission_<tag>_<n>.csv (n = siguiente número libre) y lo anota en el log.

    table: tabla de resultados donde están las métricas del modelo (results o tabpfn_results).
    """
    pred = np.clip(np.asarray(pred, dtype=float), 0, 100)
    sub = pd.DataFrame({"track_id": test["track_id"].to_numpy(), "popularity": pred})
    # Comprobaciones contra el formato de sample_submission
    assert list(sub.columns) == list(sample.columns)
    assert len(sub) == len(sample)
    assert set(sub["track_id"]) == set(sample["track_id"])
    assert sub["popularity"].notna().all()

    tag = TAGS[name]
    pattern = re.compile(rf"submission_{re.escape(tag)}_(\d+)\.csv")
    used = [int(m.group(1)) for f in OUTPUTS.iterdir() if (m := pattern.fullmatch(f.name))]
    path = OUTPUTS / f"submission_{tag}_{max(used, default=0) + 1}.csv"
    sub.to_csv(path, index=False)

    log_row = pd.DataFrame([{
        "archivo": path.name, "modelo": name,
        "CV_RMSE": table.loc[name, "CV_RMSE (train_train)"],
        "RMSE_train_test": table.loc[name, "RMSE (train_test)"],
        "fecha": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "params": str(tuned[name]["params"]) if name in tuned else "",
    }])
    log = OUTPUTS / "submissions_log.csv"
    log_row.to_csv(log, mode="a", header=not log.exists(), index=False)
    print(f"Guardado: {path}")
    return path

FINAL_RF_TREES = 1000

def final_clone(name):
    """Mismo pipeline y mismos hiperparámetros, sin entrenar; RandomForest con 1000 árboles."""
    pipe = clone(tuned[name]["pipe"])
    if name == "RandomForest":
        pipe.set_params(model__n_estimators=FINAL_RF_TREES)
    elif name == "Ensemble" and "RandomForest" in tuned["Ensemble"]["params"]["members"]:
        pipe.set_params(RandomForest__model__n_estimators=FINAL_RF_TREES)
    return pipe

best_cv = results["CV_RMSE (train_train)"].idxmin()
print("Modelo final (menor RMSE de CV):", best_cv, tuned[best_cv]["params"])

test_preds = {}                                             # nombre -> predicción sobre test
test_preds[best_cv] = final_clone(best_cv).fit(X, y).predict(test)   # reentrenar con todo train
saved = {best_cv: save_submission(test_preds[best_cv], best_cv, results)}
upload = saved[best_cv]
print(f"\n>>> Archivo para subir a Kaggle: {upload}")
pd.read_csv(upload).describe()

Modelo final (menor RMSE de CV): Ensemble {'members': ['XGBoost', 'LightGBM', 'CatBoost'], 'weights': {'XGBoost': np.float64(0.516), 'LightGBM': np.float64(0.246), 'CatBoost': np.float64(0.239)}}


Guardado: outputs/submission_ens_1.csv

>>> Archivo para subir a Kaggle: outputs/submission_ens_1.csv


,popularity
count,16268.000000
mean,37.383959
std,16.174199
min,0.000000
25%,24.477061
50%,37.498539
75%,48.801585
max,89.098595


## 6b. Una submission por cada modelo

Reentrena **cada** modelo restante con todo `train` (mismos hiperparámetros; RandomForest con 1000 árboles) y
guarda su submission en `outputs/`. Sirve para comparar en Kaggle los modelos por separado. Tarda unos
20–40 minutos; con `SAVE_EACH_MODEL = False` se omite.

In [11]:
SAVE_EACH_MODEL = True

if SAVE_EACH_MODEL:
    for name in tuned:
        if name in test_preds:           # ya guardado en la celda anterior
            continue
        t0 = time.time()
        test_preds[name] = final_clone(name).fit(X, y).predict(test)
        saved[name] = save_submission(test_preds[name], name, results)
        print(f"{'':10s}{name} ({time.time() - t0:.0f} s)")

print(f"\n>>> Archivo para subir a Kaggle: {upload}")
pd.read_csv(OUTPUTS / "submissions_log.csv")

Guardado: outputs/submission_lr_1.csv
          LinearRegression (2 s)


Guardado: outputs/submission_dt_1.csv
          DecisionTree (26 s)


Guardado: outputs/submission_lgbm_1.csv
          LightGBM (46 s)


Guardado: outputs/submission_cat_1.csv
          CatBoost (12 s)


Guardado: outputs/submission_rf_1.csv
          RandomForest (96 s)


Guardado: outputs/submission_xgb_1.csv
          XGBoost (25 s)

>>> Archivo para subir a Kaggle: outputs/submission_ens_1.csv


,archivo,modelo,CV_RMSE,RMSE_train_test,fecha,params
0,submission_ens_1.csv,Ensemble,8.098877,7.918787,2026-11-22 12:54,"{'members': ['XGBoost', 'LightGBM', 'CatBoost'..."
1,submission_lr_1.csv,LinearRegression,11.511759,11.292917,2026-11-22 12:54,{}
2,submission_dt_1.csv,DecisionTree,9.416970,9.083975,2026-11-22 12:55,"{'model__max_depth': 11, 'model__max_features'..."
3,submission_lgbm_1.csv,LightGBM,8.195477,8.015117,2026-11-22 12:55,{'model__colsample_bytree': np.float64(0.89631...
4,submission_cat_1.csv,CatBoost,8.198764,7.994268,2026-11-22 12:56,"{'model__depth': 8, 'model__iterations': 1387,..."
5,submission_rf_1.csv,RandomForest,8.307410,8.136174,2026-11-22 12:57,"{'n_estimators': 200, 'max_features': 0.516744..."
6,submission_xgb_1.csv,XGBoost,8.122149,7.943028,2026-11-22 12:58,"{'n_estimators': 2987, 'learning_rate': 0.0056..."


## 7. TabPFN (sección aparte)

Esta sección es independiente de las anteriores: si no se ejecuta, las submissions de la sección 6 ya están completas.

**TabPFN** es un modelo preentrenado para datos tabulares: no se entrena con descenso de gradiente,
sino que recibe un conjunto de filas de ejemplo (el *contexto*) y predice condicionándose en ellas.
Usa las mismas features que los modelos de árboles.

- Contexto: `TABPFN_CONTEXT` filas elegidas al azar del conjunto de entrenamiento (más contexto = mejor,
  pero más lento: con 40 000 filas tarda unos 25 minutos en un Mac con GPU MPS).
- Se ejecuta en **un proceso aparte** (`spotify.tabpfn_runner`): en macOS, XGBoost y PyTorch cargan cada uno
  su propia copia de OpenMP y en el mismo proceso se bloquean. Las predicciones se guardan en caché
  (`experiments/.cache/`), así que volver a ejecutar la celda con los mismos datos es inmediato.
- **No tiene RMSE de CV**: harían falta 5 ejecuciones de ~25 minutos. Se evalúa solo en train_test.

**Mezcla**: `(1 − W_TABPFN) · Ensemble + W_TABPFN · TabPFN`. El peso 0.5 se eligió en la fase de
exploración mirando train_test, así que el RMSE de la mezcla en train_test es algo optimista.

Requisito: licencia de TabPFN aceptada y clave en `TABPFN_TOKEN` o `~/.cache/tabpfn/auth_token`.

### 7a. Evaluación en train_test

In [7]:
from spotify.tabpfn_runner import tabpfn_predict

TABPFN_CONTEXT = 40_000
W_TABPFN = 0.5

def tree_features(X_fit, y_fit, X_apply):
    """Features del preprocesador de árboles: cross-fitting en X_fit, transform en X_apply."""
    prep = Pipeline([("rows", ROW_FEATURES), ("prep", make_tree_preprocessor(RANDOM_STATE))])
    return prep.fit_transform(X_fit, y_fit), prep.transform(X_apply)

A_tt, A_th = tree_features(X_tt, y_tt, X_th)
p_tabpfn_th = tabpfn_predict(A_tt, y_tt, A_th, n_context=TABPFN_CONTEXT, random_state=RANDOM_STATE)
p_ens_th = tuned["Ensemble"]["pipe"].predict(X_th)
p_blend_th = (1 - W_TABPFN) * p_ens_th + W_TABPFN * p_tabpfn_th

rows = []
for name, pred in [("TabPFN", p_tabpfn_th), ("Ensemble + TabPFN", p_blend_th)]:
    pred = np.clip(pred, 0, 100)
    rows.append({
        "model": name,
        "CV_RMSE (train_train)": np.nan,          # sin CV (ver nota de arriba)
        "RMSE (train_test)": root_mean_squared_error(y_th, pred),
        "MAE (train_test)": mean_absolute_error(y_th, pred),
        "R2 (train_test)": r2_score(y_th, pred),
    })
tabpfn_results = pd.DataFrame(rows).set_index("model")

# Comparación con el modelo final de la sección 6
pd.concat([results.loc[[best_cv, "Ensemble"]].drop_duplicates(), tabpfn_results]).round(4)

  TabPFN (mps, contexto 40000): 2000/13015 filas (152 s)


  TabPFN (mps, contexto 40000): 4000/13015 filas (345 s)


  TabPFN (mps, contexto 40000): 6000/13015 filas (520 s)


  TabPFN (mps, contexto 40000): 8000/13015 filas (704 s)


  TabPFN (mps, contexto 40000): 10000/13015 filas (884 s)


  TabPFN (mps, contexto 40000): 12000/13015 filas (1061 s)


  TabPFN (mps, contexto 40000): 13015/13015 filas (1236 s)


TabPFN: 13015 predicciones en 1241 s


,CV_RMSE (train_train),RMSE (train_test),MAE (train_test),R2 (train_test)
model,,,,
Ensemble,8.0989,7.9188,5.0596,0.8399
TabPFN,NaN,7.8229,4.8951,0.8437
Ensemble + TabPFN,NaN,7.7987,4.9310,0.8447


### 7b. TabPFN sobre test y mezcla con el ensamble

TabPFN usa como contexto `TABPFN_CONTEXT` filas de **todo** `train`. La mezcla usa el ensamble reentrenado con
todo `train` en la sección 6 (si no se reentrenó allí, se reentrena aquí).

Se recomienda subir la mezcla solo si su RMSE en train_test es menor que el del modelo final de la sección 6.

In [8]:
A_full, A_test = tree_features(X, y, test)
p_tabpfn_test = tabpfn_predict(A_full, y, A_test, n_context=TABPFN_CONTEXT, random_state=RANDOM_STATE)

if "Ensemble" not in test_preds:
    test_preds["Ensemble"] = final_clone("Ensemble").fit(X, y).predict(test)
test_preds["TabPFN"] = p_tabpfn_test
test_preds["Ensemble + TabPFN"] = (1 - W_TABPFN) * test_preds["Ensemble"] + W_TABPFN * p_tabpfn_test

for name in ["TabPFN", "Ensemble + TabPFN"]:
    saved[name] = save_submission(test_preds[name], name, tabpfn_results)

rmse_blend = tabpfn_results.loc["Ensemble + TabPFN", "RMSE (train_test)"]
rmse_best = results.loc[best_cv, "RMSE (train_test)"]
use_blend = rmse_blend < rmse_best
print(f"\nRMSE train_test: Ensemble + TabPFN {rmse_blend:.4f} vs {best_cv} {rmse_best:.4f}")
upload = saved["Ensemble + TabPFN"] if use_blend else saved[best_cv]
print(f">>> Archivo para subir a Kaggle: {upload}")
pd.read_csv(OUTPUTS / "submissions_log.csv")

  TabPFN (mps, contexto 40000): 2000/16268 filas (168 s)


  TabPFN (mps, contexto 40000): 4000/16268 filas (356 s)


  TabPFN (mps, contexto 40000): 6000/16268 filas (546 s)


  TabPFN (mps, contexto 40000): 8000/16268 filas (730 s)


  TabPFN (mps, contexto 40000): 10000/16268 filas (921 s)


  TabPFN (mps, contexto 40000): 12000/16268 filas (1106 s)


... (2 outputs left out)

  TabPFN (mps, contexto 40000): 16268/16268 filas (1667 s)


TabPFN: 16268 predicciones en 1671 s


Guardado: outputs/submission_tabpfn_1.csv
Guardado: outputs/submission_ens_tabpfn_1.csv

RMSE train_test: Ensemble + TabPFN 7.7987 vs Ensemble 7.9188
>>> Archivo para subir a Kaggle: outputs/submission_ens_tabpfn_1.csv


,archivo,modelo,CV_RMSE,RMSE_train_test,fecha,params
0,submission_ens_1.csv,Ensemble,8.098877,7.918787,2026-11-22 12:54,"{'members': ['XGBoost', 'LightGBM', 'CatBoost'..."
1,submission_lr_1.csv,LinearRegression,11.511759,11.292917,2026-11-22 12:54,{}
2,submission_dt_1.csv,DecisionTree,9.416970,9.083975,2026-11-22 12:55,"{'model__max_depth': 11, 'model__max_features'..."
3,submission_lgbm_1.csv,LightGBM,8.195477,8.015117,2026-11-22 12:55,{'model__colsample_bytree': np.float64(0.89631...
4,submission_cat_1.csv,CatBoost,8.198764,7.994268,2026-11-22 12:56,"{'model__depth': 8, 'model__iterations': 1387,..."
5,submission_rf_1.csv,RandomForest,8.307410,8.136174,2026-11-22 12:57,"{'n_estimators': 200, 'max_features': 0.516744..."
6,submission_xgb_1.csv,XGBoost,8.122149,7.943028,2026-11-22 12:58,"{'n_estimators': 2987, 'learning_rate': 0.0056..."
7,submission_tabpfn_1.csv,TabPFN,NaN,7.822884,2026-11-22 14:00,NaN
8,submission_ens_tabpfn_1.csv,Ensemble + TabPFN,NaN,7.798731,2026-11-22 14:00,NaN
